# 01 · Data cleaning and exploratory analysis

**Goal:** understand the data, define the claim target, choose predictors, and document every cleaning decision.

Sections:
1. Load and inspect
2. Understand the variables
3. Define the target
4. Clean and create features
5. Explore claim rates by risk factor
6. Choose predictors and save train/test files

In [28]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/

import numpy as np
import pandas as pd
from lets_plot import *

from src import config

LetsPlot.setup_html()

## 1. Load and inspect

In [29]:
from src.data_prep import load_raw
raw = load_raw()
print(raw.shape)
raw.head()

(354140, 47)


,insured_id,year,policy_type,policy_status,business_type,payment_frequency,bonus_score,driver_age,vehicle_age,age_driving_licence,...,liability_property_incurred,liability_injury_incurred,property_incurred,theft_incurred,fire_incurred,glass_incurred,legal_protection_incurred,occupants_incurred,total_exposure,liability_exposure
0,1,2022,COMP_E,C,NB,A,G,48,22.0,26.0,...,0.000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.09863,0.09863
1,2,2022,COMP_E,A,P,A,G,43,24.0,19.0,...,0.000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000
2,2,2023,COMP_E,A,P,A,G,44,25.0,19.0,...,233.013,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000
3,2,2024,COMP_E,A,P,A,G,45,26.0,18.0,...,1035.000,0.0,1551.3615,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000
4,3,2022,COMP_E,A,NB,S,G,45,26.0,19.0,...,0.000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,1.00000,1.00000


In [30]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 354140 entries, 0 to 354139
Data columns (total 47 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   insured_id                   354140 non-null  int64  
 1   year                         354140 non-null  int64  
 2   policy_type                  354140 non-null  str    
 3   policy_status                354140 non-null  str    
 4   business_type                354140 non-null  str    
 5   payment_frequency            354140 non-null  str    
 6   bonus_score                  354140 non-null  str    
 7   driver_age                   354140 non-null  int64  
 8   vehicle_age                  354138 non-null  float64
 9   age_driving_licence          354138 non-null  float64
 10  fuel_type                    352853 non-null  str    
 11  vehicle_value                353627 non-null  float64
 12  seats                        354140 non-null  int64  
 13  power_to_w

In [31]:
# Missing values per column
raw.isna().mean().sort_values(ascending=False).head(20)

fuel_type                  0.003634
vehicle_value              0.001449
age_driving_licence        0.000006
vehicle_age                0.000006
insured_id                 0.000000
year                       0.000000
policy_type                0.000000
bonus_score                0.000000
payment_frequency          0.000000
business_type              0.000000
policy_status              0.000000
driver_age                 0.000000
seats                      0.000000
power_to_weight_ratio      0.000000
vehicle_brand              0.000000
municipality_type          0.000000
circulation_area           0.000000
total_premium              0.000000
liability_premium          0.000000
property_damage_premium    0.000000
dtype: float64

## 2. Understand the variables

The file contains policy years **2022, 2023, and 2024** only. Train on 2022–2023 and test on 2024. That 2024 test is the stand-in for “next year.” The app can later apply the same model to a new driver and report a next-year claim probability. There is no 2026 outcome in this file, so 2026 is not the training target.

For each row, a column is a fair predictor only if the insurer would have known it at the **start of that row’s policy year**.

> ⚠️ Only use information available **before** the year you are predicting. Variables recorded after a claim (e.g. claim costs, cancellations caused by claims) would leak the answer.

### How these predictors were chosen

Definitions come from `data/raw/Descriptive of variables.csv`. The 14 columns below are a **candidate list**, chosen before looking at claim rates. A column was included only when both of these were true:

1. The insurer would already know it on the first day of that policy year.
2. It describes the driver, the vehicle, or the contract, so it could plausibly affect claim risk.

No column was kept or dropped because of a chart or a correlation. That check is section 5. After those claim-rate plots, keep the candidates whose rate actually changes across groups and drop the ones that do not. `src/config.py` still has placeholder feature names until that choice is made.

Columns failed the rule for a specific reason:

- `total_claims` and the other `*_claims` and `*_incurred` columns are the outcome for that same year.
- `policy_status` can change during the year, including after a claim.
- Premium columns are the insurer’s current price, so using them would mostly copy the existing rating.
- `insured_id` and `year` only identify the policy and split 2022–2023 from 2024.
- `total_exposure` is how much of the year the policy was in force. It is an offset, not a reason the driver is risky.

### Predictors

Known at the start of the policy year. These describe the driver, the vehicle, and the contract.

| Variable | Meaning | Type |
|---|---|---|
| `driver_age` | Age of the main driver, in years | numeric |
| `vehicle_age` | Age of the insured vehicle, in years | numeric |
| `age_driving_licence` | Years since the driving licence was issued. Values run from 0 to 80 (median 20), so this is a duration, not a calendar year | numeric |
| `vehicle_value` | Declared insured value of the vehicle | numeric |
| `seats` | Number of seats | numeric |
| `power_to_weight_ratio` | Vehicle weight per horsepower, a proxy for performance | numeric |
| `fuel_type` | Diesel (`D`) or gasoline (`G`) | category |
| `vehicle_brand` | Brand of the insured vehicle | category |
| `policy_type` | Coverage package: `TP` third party, `TPG` third party plus glass, `CC` combined cover, `COMP_E` comprehensive with excess, `COMP_N` comprehensive without excess | category |
| `business_type` | `NB` new business or `P` existing portfolio policy | category |
| `payment_frequency` | Billing frequency: `A` annual, `S` semiannual, `Q` quarterly | category |
| `bonus_score` | Past claims history: `G` good, `N` neutral, `B` bad | category |
| `municipality_type` | `I` inland, `C` coastal, `IS` islands | category |
| `circulation_area` | Usual driving area: `U` urban or `R` rural | category |

### Identifiers and model structure

| Variable | Role |
|---|---|
| `insured_id` | Links the same policy across years. Not a predictor. |
| `year` | Used only to split 2022–2023 (train) from 2024 (test). Not a predictor. |
| `total_exposure` | Fraction of the year the policy was in force, from 0 to 1. Used as an offset so a policy covered for part of the year is not treated like a full year. Not a reason the driver is risky. |

### Left out

Same-year claim counts and costs are the outcome. That includes `total_claims` and every `*_claims` and `*_incurred` column.

`policy_status` records whether the policy was still active (`A`) or was cancelled (`C`) during that year. A cancellation can follow a claim, so it is not known at the start of the year.

Premium columns (`total_premium` and each `*_premium`) are the insurer’s existing price. Using them would mostly copy the current rating instead of learning risk from the driver and the vehicle.

## 3. Define the target

**Choice:** all coverages, using `total_claims`.

The question is whether a policy has at least one claim in the year, not a claim under one coverage only. `total_claims` already adds every coverage together. The model target is `claim = 1` when `total_claims > 0`.

About 86% of policy-years have zero claims, and about 14% have one or more (the maximum is 18). The count is mostly zeros, so a yes/no target matches the question better than predicting the exact number of claims.

In [35]:
counts = raw["total_claims"].value_counts().sort_index()
summary = pd.DataFrame({"policies": counts, "share": counts / counts.sum()})
summary.index.name = "total_claims"
print(summary.to_string())
print(f"\nPolicies with at least one claim: {(raw['total_claims'] > 0).mean():.1%}")

by_year = raw.groupby("year").apply(
    lambda d: pd.Series({
        "policies": len(d),
        "at_least_one_claim": (d["total_claims"] > 0).mean(),
    }),
    include_groups=False,
)
print("\nBy policy year")
print(by_year.to_string())



              policies     share
total_claims                    
0               303333  0.856534
1                35810  0.101118
2                 8902  0.025137
3                 3212  0.009070
4                 1455  0.004109
5                  912  0.002575
6                  323  0.000912
7                  112  0.000316
8                   38  0.000107
9                   22  0.000062
10                   8  0.000023
11                   7  0.000020
13                   3  0.000008
14                   1  0.000003
17                   1  0.000003
18                   1  0.000003

Policies with at least one claim: 14.3%

By policy year
      policies  at_least_one_claim
year                              
2022   67172.0            0.129876
2023  118835.0            0.139984
2024  168133.0            0.151356


In [36]:
plot_counts = raw["total_claims"].clip(upper=5).value_counts().sort_index()
plot_df = plot_counts.rename_axis("claims").reset_index(name="policies")
plot_df["claims"] = plot_df["claims"].map(lambda i: str(int(i)) if i < 5 else "5+")

ggplot(plot_df, aes(x="claims", y="policies")) + \
    geom_bar(stat="identity", fill="#4C78A8") + \
    labs(
        x="Number of claims in the policy year",
        y="Policies",
        title="Claim count distribution, all coverages",
    ) + \
    ggsize(700, 400)

## 4. Clean and create features

TODO: implement `clean()` and `add_features()` in `src/data_prep.py`, then run them here.

In [37]:
import importlib
importlib.reload(config)

from src.data_prep import clean, add_features, make_target
df = make_target(add_features(clean(raw)))
df.shape

(354140, 48)

## 5. Explore claim rates by risk factor

Use **exposure-weighted** claim rates: total claims ÷ total exposure per group. A simple mean of the 0/1 target ignores that some policies were only insured for part of the year.

In [38]:
def claim_rate_by(df, col, bins=None):
    g = df[col] if bins is None else pd.cut(df[col], bins)
    out = df.groupby(g, observed=True).agg(
        policies=(config.TARGET, "size"),
        exposure=(config.EXPOSURE_COL, "sum"),
        claims=(config.TARGET, "sum"),
    )
    out["claims_per_exposure_year"] = out["claims"] / out["exposure"]
    return out

# TODO: run for each candidate predictor and plot the most interesting ones
# claim_rate_by(df, "driver_age", bins=[17, 25, 35, 45, 55, 65, 100])

**Observations:**

TODO: write 1–2 sentences per chart. What pattern do you see and does it make sense?

## 6. Choose predictors and save

TODO: update `FEATURES` in `src/config.py`, then run `python -m src.data_prep` (or `main()` below).

In [39]:
from src.data_prep import main
main()

Raw data: 354,140 rows, 47 columns
Train: 186,007 rows | claim rate 13.63%
Test:  168,133 rows | claim rate 15.14%
Saved to C:\Users\angek\OneDrive\Others\Bureau\Senior-Project\predictNextClaim\data\processed
